# Análisis Exploratorio de Datos (EDA) Inicial - ODEPA

Este notebook contiene la estructura inicial para el diagnóstico y exploración del dataset de precios hortofrutícolas de la Oficina de Estudios y Políticas Agrarias (ODEPA).

> **Nota Importante:** En esta primera fase, **no se aplica ningún tipo de normalización de datos ni escalado de variables**. El objetivo principal es realizar un diagnóstico empírico de la calidad del dataset, comprender su estructura básica, analizar la cardinalidad de sus variables categóricas, rastrear registros con valores en cero y extraer componentes temporales básicos.

## ️ Configuración de la Terminal (Fish Shell)

Si estás utilizando la terminal **Fish Shell** en Linux, puedes configurar tu entorno de la siguiente manera:

1. **Crear entorno virtual:**
   ```fish
   python3 -m venv .venv
   ```
2. **Activar en Fish:**
   ```fish
   source .venv/bin/activate.fish
   ```
3. **Instalar dependencias:**
   ```fish
   pip install -r requirements.txt
   ```
4. **Registrar kernel de Jupyter:**
   ```fish
   python3 -m ipykernel install --user --name=prediccion-precios --display-name "Predicción Precios (ODEPA)"
   ```
5. **Ejecutar Jupyter:**
   ```fish
   jupyter notebook
   ```

## 1. Carga del Dataset

En esta celda importamos las librerías necesarias (`pandas`, `numpy`, `matplotlib`, `seaborn`) y realizamos la lectura inicial del archivo `dataset/2025.csv` sin realizar modificaciones en sus tipos de datos originales.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ruta al dataset del año 2025
csv_path = 'dataset/2025.csv'

# Carga básica del CSV
print(f"Cargando archivo: {csv_path}...")
df = pd.read_csv(csv_path)
print("Carga completa.")

Cargando archivo: dataset/2025.csv...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/2025.csv'

## 2. Diagnóstico Estructural del Dataset

Evaluamos la forma (dimensiones), los tipos de datos de cada columna y visualizamos las primeras filas para comprender el contenido bruto del dataset.

In [ ]:
# 1. Dimensiones del dataset
print(f"--- Dimensiones del Dataset ---")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}\n")

# 2. Tipos de datos e información estructural
print("--- Información del DataFrame ---")
df.info()

# 3. Muestra de los primeros registros
print("\n--- Primeras 5 filas del Dataset ---")
df.head()

--- Dimensiones del Dataset ---
Filas: 202029
Columnas: 15

--- Información del DataFrame ---
<class 'pandas.DataFrame'>
RangeIndex: 202029 entries, 0 to 202028
Data columns (total 15 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   _id                         202029 non-null  int64
 1   Fecha                       202029 non-null  str  
 2   ID region                   202029 non-null  int64
 3   Region                      202029 non-null  str  
 4   Mercado                     202029 non-null  str  
 5   Subsector                   202029 non-null  str  
 6   Producto                    202029 non-null  str  
 7   Variedad / Tipo             202029 non-null  str  
 8   Calidad                     202029 non-null  str  
 9   Unidad de comercializacion  202029 non-null  str  
 10  Origen                      202029 non-null  str  
 11  Volumen                     202029 non-null  int64
 12  Precio minimo    

,_id,Fecha,ID region,Region,Mercado,Subsector,Producto,Variedad / Tipo,Calidad,Unidad de comercializacion,Origen,Volumen,Precio minimo,Precio maximo,Precio promedio
0,1,2025-01-01,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Hortalizas y tubérculos,Ají,Amarillo,Extra,$/caja 12 kilos,Región de Arica y Parinacota,0,"0,0000","0,0000","0,0000"
1,2,2025-01-01,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Ajo,Rosado,1a (cosecha),$/trenza 50 unidades,Región de Arica y Parinacota,0,"0,0000","0,0000","0,0000"
2,3,2025-01-01,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Ajo,Chino,1a (cosecha),$/trenza 50 unidades,Región de Arica y Parinacota,0,"0,0000","0,0000","0,0000"
3,4,2025-01-01,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Ajo,Rosado,2a (cosecha),$/trenza 50 unidades,Región de Arica y Parinacota,0,"0,0000","0,0000","0,0000"
4,5,2025-01-01,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Ajo,Chino,2a (cosecha),$/trenza 50 unidades,Región de Arica y Parinacota,0,"0,0000","0,0000","0,0000"


## 3. Análisis de Ceros (Valores Vacíos o Nulos Implícitos)

En datasets agrícolas, los registros con `Precio promedio` o `Volumen` igual a cero o con el texto `'0,0000'` suelen indicar la ausencia de transacciones comerciales en el mercado (por ejemplo, días feriados o problemas de registro). Identificamos la presencia de estos valores para justificar futuras estrategias de limpieza y filtrado.

In [ ]:
# Definimos las condiciones de cero para Volumen (que puede ser numérico o string)
volumen_zero_mask = (
    (df['Volumen'] == 0) | 
    (df['Volumen'] == 0.0) | 
    (df['Volumen'].astype(str).str.strip() == '0') | 
    (df['Volumen'].astype(str).str.strip() == '0,0000')
)

# Definimos las condiciones de cero para Precio promedio (que suele ser cargado como string por comas decimales)
precio_zero_mask = (
    (df['Precio promedio'] == 0) | 
    (df['Precio promedio'] == 0.0) | 
    (df['Precio promedio'].astype(str).str.strip() == '0') | 
    (df['Precio promedio'].astype(str).str.strip() == '0,0000')
)

# Conteos
volumen_zeros_count = volumen_zero_mask.sum()
precio_zeros_count = precio_zero_mask.sum()
both_zeros_count = (volumen_zero_mask & precio_zero_mask).sum()
either_zeros_count = (volumen_zero_mask | precio_zero_mask).sum()

print(f"Registros con Volumen = 0 (o '0,0000'): {volumen_zeros_count} ({volumen_zeros_count / len(df) * 100:.2f}% del total)")
print(f"Registros con Precio promedio = 0 (o '0,0000'): {precio_zeros_count} ({precio_zeros_count / len(df) * 100:.2f}% del total)")
print(f"Registros con AMBOS en cero: {both_zeros_count} ({both_zeros_count / len(df) * 100:.2f}% del total)")
print(f"Registros con AL MENOS UNO en cero: {either_zeros_count} ({either_zeros_count / len(df) * 100:.2f}% del total)\n")

# Localización de una muestra de filas afectadas
print("Muestra de filas que presentan ceros en Volumen o Precio promedio:")
cols_interes = ['_id', 'Fecha', 'Producto', 'Volumen', 'Precio promedio']
df[volumen_zero_mask | precio_zero_mask][cols_interes].head(10)

Registros con Volumen = 0 (o '0,0000'): 201 (0.10% del total)
Registros con Precio promedio = 0 (o '0,0000'): 201 (0.10% del total)
Registros con AMBOS en cero: 201 (0.10% del total)
Registros con AL MENOS UNO en cero: 201 (0.10% del total)

Muestra de filas que presentan ceros en Volumen o Precio promedio:


,_id,Fecha,Producto,Volumen,Precio promedio
0,1,2025-01-01,Ají,0,"0,0000"
1,2,2025-01-01,Ajo,0,"0,0000"
2,3,2025-01-01,Ajo,0,"0,0000"
3,4,2025-01-01,Ajo,0,"0,0000"
4,5,2025-01-01,Ajo,0,"0,0000"
5,6,2025-01-01,Ajo,0,"0,0000"
6,7,2025-01-01,Ajo,0,"0,0000"
7,8,2025-01-01,Ajo,0,"0,0000"
8,9,2025-01-01,Ajo,0,"0,0000"
9,10,2025-01-01,Ajo,0,"0,0000"


## 4. Análisis de Cardinalidad de Columnas Categóricas

La cardinalidad (número de valores únicos) determina la complejidad del preprocesamiento y la estrategia de codificación de variables categóricas. Analizamos las columnas indicadas (`Calidad`, `Subsector`, `Mercado`, `Producto`, `Variedad / Tipo`, `Origen`) para justificar técnicamente los pasos a seguir.

In [ ]:
categorical_cols = ['Calidad', 'Subsector', 'Mercado', 'Producto', 'Variedad / Tipo', 'Origen']

print(f"{'Columna':<18} | {'Valores Únicos':<15} | {'Tipo de Codificación Propuesta'}")
print("-" * 80)

for col in categorical_cols:
    if col in df.columns:
        n_unique = df[col].nunique()
        
        # Propuesta preliminar de codificación según cardinalidad
        if n_unique <= 5:
            propuesta = "One-Hot Encoding (Baja cardinalidad)"
        elif n_unique <= 15:
            propuesta = "One-Hot o Target Encoding (Cardinalidad moderada)"
        else:
            propuesta = "Target / Frequency Encoding (Alta cardinalidad, previene dimensionalidad alta)"
            
        print(f"{col:<18} | {n_unique:<15} | {propuesta}")
    else:
        print(f"{col:<18} | Columna no encontrada en el DataFrame")

print("-" * 80)

Columna            | Valores Únicos  | Tipo de Codificación Propuesta
--------------------------------------------------------------------------------
Calidad            | 54              | Target / Frequency Encoding (Alta cardinalidad, previene dimensionalidad alta)
Subsector          | 2               | One-Hot Encoding (Baja cardinalidad)
Mercado            | 12              | One-Hot o Target Encoding (Cardinalidad moderada)
Producto           | 81              | Target / Frequency Encoding (Alta cardinalidad, previene dimensionalidad alta)
Variedad / Tipo    | 242             | Target / Frequency Encoding (Alta cardinalidad, previene dimensionalidad alta)
Origen             | 65              | Target / Frequency Encoding (Alta cardinalidad, previene dimensionalidad alta)
--------------------------------------------------------------------------------


## 5. Análisis Temporal (Fechas y Patrones Básicos)

Convertimos la columna `Fecha` a tipo datetime y extraemos componentes clave como el mes y el día de la semana. Estos componentes serán fundamentales en etapas posteriores para capturar la estacionalidad (patrones mensuales y semanales).

In [ ]:
# Clonamos el DataFrame para no mutar el original en esta fase exploratoria pura
df_temp = df.copy()

# 1. Conversión de columna Fecha a tipo datetime
df_temp['Fecha'] = pd.to_datetime(df_temp['Fecha'])

# 2. Extracción del mes
df_temp['Mes'] = df_temp['Fecha'].dt.month

# 3. Extracción del día de la semana (0=Lunes, 6=Domingo)
df_temp['Dia_Semana'] = df_temp['Fecha'].dt.dayofweek

# Mapeo de día de la semana a su nombre en español para legibilidad independiente del sistema operativo
dias_map = {
    0: 'Lunes',
    1: 'Martes',
    2: 'Miércoles',
    3: 'Jueves',
    4: 'Viernes',
    5: 'Sábado',
    6: 'Domingo'
}
df_temp['Dia_Semana_Nombre'] = df_temp['Dia_Semana'].map(dias_map)

print("Dimensiones tras añadir componentes de fecha:", df_temp.shape)
print("\nMuestra del comportamiento temporal:")
df_temp[['Fecha', 'Mes', 'Dia_Semana', 'Dia_Semana_Nombre']].head(10)

Dimensiones tras añadir componentes de fecha: (202029, 18)

Muestra del comportamiento temporal:


,Fecha,Mes,Dia_Semana,Dia_Semana_Nombre
0,2025-01-01,1,2,Miércoles
1,2025-01-01,1,2,Miércoles
2,2025-01-01,1,2,Miércoles
3,2025-01-01,1,2,Miércoles
4,2025-01-01,1,2,Miércoles
5,2025-01-01,1,2,Miércoles
6,2025-01-01,1,2,Miércoles
7,2025-01-01,1,2,Miércoles
8,2025-01-01,1,2,Miércoles
9,2025-01-01,1,2,Miércoles
